# 15.2 协同过滤 / Collaborative Filtering

**中文**：上一节基于内容只用了"物品自身的特征"，结果输给了热门基线。本节我们换一种完全不同的思路——**协同过滤（Collaborative Filtering, CF）**：*"不看物品长什么样，只看用户的行为。和你口味相近的人喜欢的东西，你大概也会喜欢。"* 这是推荐系统真正的起点，也是 Netflix Prize、Amazon 推荐的核心。
**English**: Content-based used only "item features" and lost to the popularity baseline. Now a completely different idea — **Collaborative Filtering (CF)**: *"ignore what the item looks like; look only at user behavior. People with taste similar to yours liked X, so you probably will too."* This is the true birthplace of recommender systems and the core of the Netflix Prize and Amazon's recommendations.

---

**中文**：CF 有两大流派，本节都手写实现：
1. **基于用户（user-user）**：找和你最像的一批用户，把他们喜欢的东西推给你。
2. **基于物品（item-item）**：找和你喜欢过的物品最像的物品（"喜欢这个的人也喜欢那个"）。注意这里的"物品相似"不是看类型标签，而是看**被同一批人喜欢**——是行为定义的相似，不是内容定义的。

**English**: CF has two schools, both hand-coded here:
1. **User-user**: find users most similar to you, recommend what they liked.
2. **Item-item**: find items most similar to those you liked ("people who liked this also liked that"). Crucially, "item similarity" here is *not* genre tags but **being liked by the same people** — behavioral similarity, not content similarity.

> 💡 **面试速查 / Interview cheat-sheet（★★★ 极高频）**
> **中文**：**user-user vs item-item**——工业界几乎都用 **item-item**，原因：① 用户数 >> 物品数时物品相似度矩阵更小、更稳定；② 物品相似度随时间变化慢（可离线预计算），用户口味变化快；③ 可解释（"因为你买过 X"）。著名落地：Amazon 的 "item-to-item collaborative filtering"。**核心痛点**：**稀疏性**（评分矩阵 99%+ 是空的）和**冷启动**（新用户/新物品没有行为）。
> **English**: **User-user vs item-item** — industry overwhelmingly uses **item-item** because: ① when #users >> #items the item-item matrix is smaller and more stable; ② item similarities change slowly (precompute offline) while user taste shifts fast; ③ explainable ("because you bought X"). Famous deployment: Amazon's "item-to-item collaborative filtering." **Key pains**: **sparsity** (the rating matrix is 99%+ empty) and **cold start** (new users/items have no behavior).


In [ ]:

# ============================================================
# 数据：构造用户-物品评分矩阵 / Build the user-item rating matrix
# 中文：CF 的核心数据结构是一个 (用户 × 物品) 的评分矩阵 R，R[u,i] 是用户u对物品i的评分，
#       绝大多数是缺失(0)。MovieLens-100k 的矩阵 943×1682 中只有 6.3% 有评分 —— 这就是"稀疏性"。
# English: CF's core structure is a (users × items) rating matrix R; R[u,i] is u's rating of i,
#          mostly missing (0). For ML-100k (943×1682) only 6.3% of cells are filled — sparsity.
# ============================================================
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
np.random.seed(0)
R_DIR = os.path.expanduser("~/.cache/dsfs_recsys/ml-100k")
ratings = pd.read_csv(os.path.join(R_DIR,"u.data"), sep="\t",
                      names=["user","item","rating","ts"])

# 按时间给每个用户切 80/20：早期训练、近期测试（模拟"用过去预测未来"）
# Temporal 80/20 split per user: train on earlier, test on later ratings.
rs = ratings.sort_values("ts")
tr_parts, te_parts = [], []
for _, g in rs.groupby("user"):
    c = int(len(g)*0.8)
    tr_parts.append(g.iloc[:c]); te_parts.append(g.iloc[c:])
train = pd.concat(tr_parts); test = pd.concat(te_parts)

nU = ratings["user"].max()+1     # 用户 id 从 1 开始，留 0 位方便用 id 直接当下标
nI = ratings["item"].max()+1     # 物品 id 同理 / item ids start at 1
Rm = np.zeros((nU, nI))          # 稠密评分矩阵 (944,1683)，未评分为 0 / dense matrix, 0 = unrated
for u,i,r in zip(train["user"], train["item"], train["rating"]):
    Rm[u, i] = r                 # 填入训练评分 / fill training ratings
mask = Rm > 0                    # 布尔掩码：哪些位置有评分 / which cells are observed

density = mask.sum() / (nU*nI)
print(f"矩阵 / matrix: {Rm.shape}, 训练评分 train ratings: {int(mask.sum())}")
print(f"稠密度 density: {density:.3%}  (即 {1-density:.1%} 是空的 / empty)")


**中文**：直接对原始评分算相似度有个陷阱：有人是"宽容打分者"（什么都给 4-5 星），有人苛刻（很少给高分）。所以要先做**按用户均值中心化**——每个评分减去该用户自己的平均分，得到"相对于他自己平均水平的偏好"。
**English**: Computing similarity on raw ratings has a trap: some users are lenient (everything is 4-5 stars), others harsh. So we **mean-center per user** — subtract each user's own average from their ratings, yielding "preference relative to that user's own baseline."

**中文**：中心化之后，用户之间用**余弦相似度**（等价于把缺失当 0 的皮尔逊相关）衡量口味接近程度。
**English**: After centering, we measure taste closeness between users with **cosine similarity** (equivalent to Pearson correlation when treating missing as 0).


In [ ]:

# ============================================================
# user-user CF：用户相似度 + 加权预测 / user similarity + weighted prediction
# ============================================================
# 每个用户的均值（只对已评分的算）/ per-user mean over observed ratings only
user_mean = np.array([Rm[u, mask[u]].mean() if mask[u].any() else 0.0 for u in range(nU)])
Rc = np.where(mask, Rm - user_mean[:,None], 0.0)   # 中心化矩阵，未评分仍为0 / centered, unrated=0

# 用户两两余弦相似度 / pairwise user cosine similarity
unorm = np.linalg.norm(Rc, axis=1, keepdims=True); unorm[unorm==0]=1e-9
Un = Rc / unorm                                    # 行归一化 / row-normalized
Suu = Un @ Un.T                                    # (nU,nU) 用户相似度矩阵 / user-user sim

def predict_uu(u, i, k=30):
    """中文：用最相似的k个'也评过物品i'的邻居，加权预测u对i的评分。
       English: predict u's rating of i from the k most similar neighbors who rated i."""
    raters = np.where(mask[:, i])[0]               # 评过物品 i 的所有用户 / users who rated i
    raters = raters[raters != u]                   # 排除自己 / exclude u
    if len(raters)==0: return user_mean[u]         # 没人评过就回退到用户均值 / fallback
    sims = Suu[u, raters]                           # u 与这些邻居的相似度
    top = np.argsort(sims)[::-1][:k]                # 取相似度最高的 k 个 / top-k neighbors
    nb, w = raters[top], sims[top]
    if np.abs(w).sum() < 1e-8: return user_mean[u]
    # 预测 = 用户均值 + 邻居"中心化评分"的相似度加权平均 / mean + sim-weighted neighbor deviations
    return user_mean[u] + (w @ Rc[nb, i]) / np.abs(w).sum()

# 抽几条测试评分看看预测 / sanity check on a few test ratings
sample = test.sample(5, random_state=1)
print("user item  true  pred(uu)")
for u,i,r in zip(sample["user"], sample["item"], sample["rating"]):
    print(f"{u:5d}{i:5d}{r:6.1f}  {predict_uu(u,i):.2f}")


**中文**：item-item CF 把视角转 90°——不算"用户像不像"，而是算"物品像不像"。两个物品相似，当且仅当**被同一批用户给出相似的评分**。预测用户 $u$ 对物品 $i$ 的分数时，用 $u$ 评过的、与 $i$ 最相似的物品来加权。
**English**: Item-item CF rotates the view 90° — instead of "how similar are users," compute "how similar are items." Two items are similar iff **rated similarly by the same users**. To predict $u$'s rating of $i$, weight over the items $u$ has rated that are most similar to $i$.

**中文**：为什么工业界偏爱它？因为物品相似度矩阵可以**离线预计算并缓存**（物品关系稳定），线上只需查表，延迟极低——这正是 Amazon 的做法。
**English**: Why does industry prefer it? The item-item similarity matrix can be **precomputed offline and cached** (item relationships are stable); online you just look it up — extremely low latency. This is exactly Amazon's approach.


In [ ]:

# ============================================================
# item-item CF：物品相似度（在中心化矩阵的"列"上算）/ item similarity over centered columns
# ============================================================
inorm = np.linalg.norm(Rc, axis=0, keepdims=True); inorm[inorm==0]=1e-9
In = Rc / inorm                                    # 列归一化 / column-normalized
Sii = In.T @ In                                    # (nI,nI) 物品相似度矩阵 / item-item sim

def predict_ii(u, i, k=30):
    """中文：用 u 评过的、与物品 i 最相似的 k 个物品加权预测。
       English: predict from the k items u rated that are most similar to i."""
    rated = np.where(mask[u])[0]                    # u 评过的所有物品 / items u rated
    rated = rated[rated != i]
    if len(rated)==0: return user_mean[u]
    sims = Sii[i, rated]                             # 物品 i 与这些物品的相似度
    top = np.argsort(sims)[::-1][:k]
    it, w = rated[top], sims[top]
    if np.abs(w).sum() < 1e-8: return user_mean[u]
    return user_mean[u] + (w @ Rc[u, it]) / np.abs(w).sum()

print("user item  true  pred(ii)")
for u,i,r in zip(sample["user"], sample["item"], sample["rating"]):
    print(f"{u:5d}{i:5d}{r:6.1f}  {predict_ii(u,i):.2f}")


**中文**：现在做**评分预测精度评估**——经典指标 **RMSE（均方根误差）**：预测分和真实分差多少。Netflix Prize 当年就是比 RMSE。我们在测试集上对比 user-user、item-item，以及一个朴素基线"永远预测该用户的均值"。
**English**: Now evaluate **rating-prediction accuracy** with the classic **RMSE (root-mean-squared error)** — how far predictions are from true ratings. The Netflix Prize was scored on RMSE. We compare user-user, item-item, and a naive "always predict the user's mean" baseline on the test set.


In [ ]:

# ============================================================
# RMSE 评估：user-user vs item-item vs 用户均值基线 / RMSE comparison
# ============================================================
def rmse(predfn, n=3000):
    s = test.sample(min(n,len(test)), random_state=2)   # 抽样加速 / subsample for speed
    err=[]
    for u,i,r in zip(s["user"], s["item"], s["rating"]):
        err.append(predfn(u,i) - r)
    return np.sqrt(np.mean(np.square(err)))

rmse_mean = rmse(lambda u,i: user_mean[u])             # 基线：预测用户均值 / baseline
rmse_uu   = rmse(lambda u,i: predict_uu(u,i,k=30))     # user-user
rmse_ii   = rmse(lambda u,i: predict_ii(u,i,k=30))     # item-item
print(f"{'方法/method':<22}{'RMSE':>8}")
print(f"{'User-mean baseline':<22}{rmse_mean:>8.4f}")
print(f"{'User-user CF (k=30)':<22}{rmse_uu:>8.4f}")
print(f"{'Item-item CF (k=30)':<22}{rmse_ii:>8.4f}")


**中文**：RMSE 衡量"预测分准不准"，但推荐的最终目标是**排序**——能不能把用户真正喜欢的排到前面。所以我们再做 **Top-N 推荐评估**，用 Precision@10 / Recall@10，并和 15.1 一样和**热门基线**对比。这里有一个**反直觉但极重要**的现象，我们用两种不同的 top-N 打分方式揭示它：
**English**: RMSE measures prediction accuracy, but recommendation ultimately cares about **ranking** — putting truly-liked items on top. So we run **Top-N evaluation** (Precision@10 / Recall@10) vs the **popularity baseline**. Here lies a **counter-intuitive but crucial** phenomenon, which we expose with two different top-N scoring schemes:

**中文**：
- **方案A（按预测评分排序）**：直接拿上面 RMSE 最优的预测分给物品排序，取前10。这是初学者最自然的想法。
- **方案B（按相似度之和排序）**：对用户**喜欢过(≥4星)**的每个物品，把"与候选物品的相似度"累加，**不做用户均值归一化**。

**English**:
- **Scheme A (rank by predicted rating)**: rank items by the very predictions that minimized RMSE, take top-10. The beginner's natural choice.
- **Scheme B (rank by sum of similarities)**: for each item the user **liked (≥4 stars)**, accumulate "similarity to the candidate item," with **no per-user-mean normalization**.


In [ ]:

# ============================================================
# Top-N 排序评估 / Top-N ranking evaluation (item-item CF vs popularity)
# 中文：对每个用户，给所有"没看过"的物品打分，取前10，看命中多少测试集中喜欢(≥4星)的。
# English: for each user, score all unseen items, take top-10, count hits among liked(>=4) test items.
# ============================================================
# 预计算物品流行度（训练集评分人数）/ precompute item popularity from train
pop = np.zeros(nI)
for i in train["item"]: pop[i]+=1
pop_order = np.argsort(pop)[::-1]                       # 人气从高到低 / by popularity

def topN_ii(u, N=10):
    rated = np.where(mask[u])[0]                        # u 训练里看过的 / seen in train
    # 用户对所有物品的 item-item 预测分（向量化）：Sii[:,rated] @ centered ratings
    sims = Sii[:, rated]                                 # (nI, |rated|)
    num = sims @ Rc[u, rated]                            # 分子：相似度加权偏差和
    den = np.abs(sims).sum(axis=1) + 1e-8               # 分母：相似度绝对值和
    score = user_mean[u] + num/den                      # 预测分 / predicted scores
    score[rated] = -1e9                                  # 屏蔽已看过 / mask seen
    return np.argsort(score)[::-1][:N]

# 方案B：对用户喜欢过(>=4星)的物品，累加候选物品与它们的相似度（不归一化）
# Scheme B: sum similarities of the candidate to each item the user liked (>=4), un-normalized
liked_train = [np.where(Rm[u]>=4)[0] for u in range(nU)]   # 每个用户训练里喜欢的物品 / liked items
def topN_simsum(u, N=10):
    lk = liked_train[u]
    if len(lk)==0: return list(topN_pop(u,N))               # 没喜欢记录则回退热门 / fallback
    score = Sii[:, lk].sum(axis=1)                          # 与所有喜欢物品的相似度之和 / sum sims
    score[mask[u]] = -1e9                                    # 屏蔽已看过 / mask seen
    return np.argsort(score)[::-1][:N]

def topN_pop(u, N=10):
    rated=set(np.where(mask[u])[0]); out=[]
    for i in pop_order:
        if i not in rated: out.append(i)
        if len(out)==N: break
    return out

# 评估三种方案 / evaluate all three schemes
liked_by_user = {u:set(g["item"][g["rating"]>=4]) for u,g in test.groupby("user")}
A_p=A_r=B_p=B_r=po_p=po_r=0.0; n=0
for u, liked in liked_by_user.items():
    if not liked or not mask[u].any(): continue
    a=set(topN_ii(u)); b=set(topN_simsum(u)); pr=set(topN_pop(u))
    A_p+=len(a&liked)/10; A_r+=len(a&liked)/len(liked)      # 方案A 预测评分 / scheme A
    B_p+=len(b&liked)/10; B_r+=len(b&liked)/len(liked)      # 方案B 相似度和 / scheme B
    po_p+=len(pr&liked)/10; po_r+=len(pr&liked)/len(liked)  # 热门 / popularity
    n+=1
print(f"评估用户数 / users: {n}")
print(f"{'方法/method':<30}{'Precision@10':>14}{'Recall@10':>12}")
print(f"{'A) CF 按预测评分排序':<30}{A_p/n:>14.4f}{A_r/n:>12.4f}")
print(f"{'B) CF 按相似度之和排序':<30}{B_p/n:>14.4f}{B_r/n:>12.4f}")
print(f"{'Popularity baseline':<30}{po_p/n:>14.4f}{po_r/n:>12.4f}")
RESULT_A=(A_p/n,A_r/n); RESULT_B=(B_p/n,B_r/n); RESULT_PO=(po_p/n,po_r/n)


**中文**：可视化能帮你建立直觉。下面画三张图：① 用户相似度分布（多数用户对彼此其实很不相似——稀疏的后果）；② 邻居数 $k$ 对 RMSE 的影响（太少噪声大、太多引入不相关邻居）；③ CF vs 热门 的排序指标对比。
**English**: Visualization builds intuition. Three plots: ① user-similarity distribution (most users are quite dissimilar — a consequence of sparsity); ② effect of neighbor count $k$ on RMSE (too few = noisy, too many = irrelevant neighbors); ③ CF vs popularity on ranking metrics.


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig, ax = plt.subplots(1,3, figsize=(15,4))

# ① 用户相似度分布（取上三角非对角）/ distribution of user-user similarities
tri = Suu[np.triu_indices(nU,1)]
ax[0].hist(tri, bins=60, color="#4C72B0", edgecolor="white")
ax[0].axvline(tri.mean(), color="red", ls="--", label=f"mean={tri.mean():.3f}")
ax[0].set_title("用户相似度分布 / user-user similarity"); ax[0].set_xlabel("cosine sim"); ax[0].legend()

# ② 邻居数 k 对 RMSE 的影响（item-item）/ RMSE vs k
ks=[5,10,20,40,80,160]
rm=[rmse(lambda u,i,kk=kk: predict_ii(u,i,k=kk), n=1500) for kk in ks]
ax[1].plot(ks, rm, "o-", color="#55A868")
ax[1].set_title("邻居数 k 对 RMSE / RMSE vs #neighbors"); ax[1].set_xlabel("k"); ax[1].set_ylabel("RMSE")
best=ks[int(np.argmin(rm))]; ax[1].axvline(best,color="red",ls="--",label=f"best k={best}"); ax[1].legend()

# ③ 排序指标对比：方案A vs 方案B vs 热门 / ranking metrics, three schemes
labels=["Precision@10","Recall@10"]; x=np.arange(2); w=0.27
ax[2].bar(x-w,[RESULT_A[0],RESULT_A[1]],w,label="A) pred-rating",color="#8C8C8C")
ax[2].bar(x  ,[RESULT_B[0],RESULT_B[1]],w,label="B) sim-sum",color="#4C72B0")
ax[2].bar(x+w,[RESULT_PO[0],RESULT_PO[1]],w,label="Popularity",color="#C44E52")
ax[2].set_xticks(x); ax[2].set_xticklabels(labels); ax[2].set_title("Top-N：A vs B vs 热门"); ax[2].legend()
plt.tight_layout(); plt.savefig("/tmp/rec02_viz.png", dpi=80); plt.show()
print("最佳邻居数 best k =", best, "| 相似度均值 mean sim =", round(tri.mean(),4))


**中文**：结果非常值得玩味，请仔细看：
**English**: The results deserve careful reading:

**中文**：
1. **RMSE 上 CF 赢了基线**——把 RMSE 从 ~1.14（均值基线）降到 ~1.0。Netflix Prize 当年百万美金就是为了把 RMSE 降 10%，可见"评分预测"本身很难。
2. **但方案A（按预测评分排序）的 Top-N 却输给了热门基线！** 这不是 bug，而是推荐系统里最著名的反直觉结论之一（Cremonesi et al., RecSys 2010）：**RMSE 最优 ≠ 排序最优**。原因：按预测评分排序时，那些只有极少人评过、却被某个高相似邻居打了高分的**冷门长尾物品**会被推到最前面——预测分虚高、方差大，命中率自然低。
3. **方案B（按相似度之和、不归一化）反超热门基线**——它本质在问"这个候选物品和你喜欢的一堆东西到底有多像"，自然偏向既相关又有一定热度的物品，鲁棒得多。这正是工业界 item-item 召回的真实形态。

**English**:
1. **CF wins on RMSE** — dropping it from ~1.14 (mean baseline) to ~1.0. The Netflix Prize paid \$1M for a 10% RMSE cut — rating prediction is genuinely hard.
2. **But Scheme A (rank by predicted rating) LOSES to popularity on Top-N!** Not a bug — it is one of the most famous counter-intuitive results in recommenders (Cremonesi et al., RecSys 2010): **RMSE-optimal ≠ ranking-optimal**. Why: ranking by predicted rating surfaces **obscure long-tail items** that few people rated but one high-similarity neighbor scored highly — inflated, high-variance predictions, hence low hit rate.
3. **Scheme B (un-normalized sum of similarities) overtakes popularity** — it essentially asks "how similar is this candidate to the bunch of things you liked," favoring items that are both relevant and reasonably popular — far more robust. This is the real shape of industrial item-item retrieval.

**中文**：另外两点：**k 不是越大越好**——RMSE 随邻居数 k 先降后升，存在最优值（邻居太少噪声大，太多引入不相关者）；**相似度分布**显示绝大多数用户对彼此几乎不相似（均值≈0.02），这就是稀疏的直接后果。
**English**: Two more notes: **bigger k is not better** — RMSE falls then rises with k, with an optimum (too few = noise, too many = irrelevant neighbors); the **similarity distribution** shows most user pairs are nearly dissimilar (mean ≈ 0.02), a direct consequence of sparsity.

> 💼 **实战视角 / Practical angle**
> **中文**：基于邻域的 CF 在中小数据上简单有效、可解释，但有两个硬伤：① **稀疏**——评分矩阵 94% 是空的，很多用户对没有共同评分，相似度不可靠；② **不可扩展**——相似度矩阵 $O(n^2)$，物品上百万时算不动。下一节 **15.3 矩阵分解** 用低维隐向量同时解决这两个问题，把推荐推向真正的现代方法。
> **English**: Neighborhood CF is simple, effective, and explainable on small/medium data, but has two weaknesses: ① **sparsity** — 94% of cells are empty, so many user pairs share no items and similarities are unreliable; ② **non-scalability** — the $O(n^2)$ similarity matrix is intractable at millions of items. Next, **15.3 Matrix Factorization** solves both with low-dimensional latent vectors, ushering in the modern era.

---
### 小结 / Summary
- **中文**：CF 只用行为不用内容；user-user 找相似的人，item-item 找相似的物（行为相似，非内容）。
- **English**: CF uses behavior not content; user-user finds similar people, item-item finds similar items (behavioral, not content, similarity).
- **中文**：先按用户均值中心化再算余弦；预测 = 用户均值 + 邻居偏差的相似度加权。
- **English**: Mean-center per user, then cosine; prediction = user mean + sim-weighted neighbor deviations.
- **中文**：工业界偏爱 item-item（可离线预计算、稳定、可解释）；两大痛点是稀疏与冷启动。
- **English**: Industry favors item-item (precomputable, stable, explainable); the two pains are sparsity and cold start.
